# Hybrid Movie Recommender

This notebook is the experiment report for a movie recommender system. The model implementations live in `src/hybrid_movie_recommender`; the notebook only loads data, initializes models, runs sweeps, compares metrics, and records the main findings.


## Environment and Package Imports

Install dependencies from the project root if needed, then import the reusable package modules. Generated predictions and saved models are written under `outputs/`, which is intentionally ignored by Git.


In [ ]:
# Optional setup from the project root:
# %pip install -r ../requirements.txt


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from hybrid_movie_recommender.data import load_project_data, split_train_validation
from hybrid_movie_recommender.metrics import evaluate_ranking_metrics, evaluate_rmse
from hybrid_movie_recommender.models.baselines import (
    ItemAverageBaseline,
    MeanHybridRating,
    PopularityRecommender,
    RandomRecommender,
)
from hybrid_movie_recommender.models.bpr import BayesianProbabilisticRanking
from hybrid_movie_recommender.models.content import ContentBasedCFWithMovies
from hybrid_movie_recommender.models.hybrid import HybridRecommender, OptimizedHybridRanking
from hybrid_movie_recommender.models.matrix_factorization import MatrixFactorizationSGD
from hybrid_movie_recommender.models.neighborhood import ItemBasedCF, UserBasedCF
from hybrid_movie_recommender.persistence import load_all_models, save_all_models
from hybrid_movie_recommender.predictions import save_predictions_to_csv
from hybrid_movie_recommender.tuning import train_and_predict_best

OUTPUT_DIR = Path('../outputs')
PREDS_DIR = OUTPUT_DIR / 'preds'
MODELS_DIR = OUTPUT_DIR / 'models'
PREDS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Dataset and Validation Split

The project uses explicit train and test rating files plus movie metadata. A deterministic validation split is created from the training ratings for hyperparameter selection and hybrid weight learning.


In [ ]:
train_full, test_data, movies = load_project_data('../data')
train_data, val_data = split_train_validation(train_full, test_size=0.2, random_state=10)

dataset_summary = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test', 'movies'],
        'rows': [len(train_data), len(val_data), len(test_data), len(movies)],
    }
)
display(dataset_summary)
print(f"Users in train: {train_data['user_id'].nunique():,}")
print(f"Items in train: {train_data['item_id'].nunique():,}")


In [ ]:
movies.head()


## Evaluation Workflow

Each model exposes `fit`, `predict_rating`, and where relevant `recommend_topk`. The shared package utilities handle RMSE, ranking metrics, hyperparameter sweeps, and prediction CSV generation.


## Content-Based Recommender

The content model represents movies with TF-IDF features over title and genre text, optionally compresses them with truncated SVD, and builds user profiles from rated movies. Separate configurations are selected for rating prediction and ranking.


In [ ]:
param_grid = {
    'tfidf_max_features': [5000, 7500],
    'svd_dim': [256, None],
    'normalize_emb': [True],
    'text_source': ['titlegenres'],
    'profile_agg': ['avg'],
    'positive_threshold': [2.75, 3]
}

content_model_best, content_predictions = train_and_predict_best(
    model_class=lambda **kw: ContentBasedCFWithMovies(movies_df=movies, **kw),
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid=param_grid,
    output_filepath='../outputs/preds/predictions_content_best.csv',
    metric='rmse'
)


### Content-Based Ranking Configuration

This sweep optimizes NDCG@10 instead of RMSE, because the best rating predictor is not necessarily the best top-k recommender.


In [ ]:
content_ndcg_model, content_ndcg_predictions = train_and_predict_best(
    model_class=lambda **kw: ContentBasedCFWithMovies(movies_df=movies, **kw),
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'tfidf_max_features': [7500, 10000],
        'svd_dim': [1024, None],
        'normalize_emb': [True],
        'text_source': ['titlegenres'],
        'profile_agg': ['avg'],
        'positive_threshold': [2]
    },
    output_filepath='../outputs/preds/predictions_content_ndcg.csv',
    metric='ndcg',
    model_name='Content-Based CF (NDCG-optimized)'
)

content_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=content_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=2.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {content_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {content_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {content_ndcg_ranking_metrics['ndcg@10']:.4f}")


## User-Based Collaborative Filtering

UserKNN predicts ratings from similar users and precomputes a full user-item prediction matrix for efficient lookup. The rating and ranking sweeps use different neighborhood settings.


In [ ]:
user_knn_model, user_knn_predictions = train_and_predict_best(
    model_class=UserBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={'k': [200], 'min_common': [10]},
    output_filepath='../outputs/preds/predictions_userknn_best.csv',
    metric='rmse'
)


### UserKNN Ranking Configuration

The ranking run evaluates candidate neighborhoods directly with Precision@10, Recall@10, and NDCG@10.


In [ ]:
user_knn_ndcg_model, user_knn_ndcg_predictions = train_and_predict_best(
    model_class=UserBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'k': [150, 250],
        'min_common': [2, 4]
    },
    output_filepath='../outputs/preds/predictions_userknn_ndcg.csv',
    metric='ndcg',
    model_name='User-Based CF (NDCG-optimized)'
)

user_knn_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=user_knn_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=2.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {user_knn_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {user_knn_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {user_knn_ndcg_ranking_metrics['ndcg@10']:.4f}")


## Item-Based Collaborative Filtering

ItemKNN computes item-item similarities from the sparse ratings matrix. It is slower for ranking evaluation here, but it provides a useful neighborhood-based comparison to UserKNN.


In [ ]:
item_knn_model, item_knn_predictions = train_and_predict_best(
    model_class=ItemBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'k': [30, 50, 70],
        'min_common': [2, 5]
    },
    output_filepath='../outputs/preds/predictions_itemknn_best.csv',
    metric='rmse'
)


### ItemKNN Ranking Configuration

This configuration optimizes NDCG@10 with a fixed neighborhood size chosen from validation experiments.


In [ ]:
print("\n" + "="*80)
print("ITEM-BASED CF: HYPERPARAMETER TUNING FOR NDCG@10")
print("="*80)

item_knn_ndcg_model, item_knn_ndcg_predictions = train_and_predict_best(
    model_class=ItemBasedCF,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'k': [60],
        'min_common': [2]
    },
    output_filepath='../outputs/preds/predictions_itemknn_ndcg.csv',
    metric='ndcg',
    model_name='Item-Based CF (NDCG-optimized)'
)


print("\n" + "="*80)
print("EVALUATING NDCG-OPTIMIZED ITEM-BASED MODEL")
print("="*80)

item_knn_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=item_knn_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=2.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {item_knn_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {item_knn_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {item_knn_ndcg_ranking_metrics['ndcg@10']:.4f}")


## Matrix Factorization

The PyTorch matrix-factorization model learns user and item embeddings with optional bias terms and L2 regularization. One sweep targets RMSE and another targets ranking quality.


In [ ]:
mf_model, mf_predictions = train_and_predict_best(
    model_class=MatrixFactorizationSGD,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [60, 80],
        'learning_rate': [0.002],
        'n_epochs': [40],
        'use_bias': [True],
        'eval_every': [5],
        'batch_size': [1024],
        'device': ['auto'],
        'l2_reg': [0.2]
    },
    output_filepath='../outputs/preds/predictions_mf_best.csv',
    metric='rmse'
)

mf_model.plot_training_history()
optimal_epochs = mf_model.get_best_epoch()

should_retrain = optimal_epochs is not None and optimal_epochs < mf_model.n_epochs

if should_retrain:
    print(f"\nRetraining with {optimal_epochs} epochs (best val: {mf_model.best_val_rmse:.4f})")

    mf_model = MatrixFactorizationSGD(
        n_factors=mf_model.n_factors,
        learning_rate=mf_model.learning_rate,
        n_epochs=optimal_epochs,
        use_bias=mf_model.use_bias,
        eval_every=mf_model.eval_every,
        batch_size=mf_model.batch_size,
        l2_reg=mf_model.l2_reg,
        device='auto'
    ).fit(train_data, val_data)

    print(f"Training complete: {optimal_epochs} epochs, Val RMSE: {mf_model.best_val_rmse:.4f}")


### Matrix Factorization Ranking Configuration

This run evaluates whether embedding factors trained with a ranking-oriented sweep improve top-k recommendation quality.


In [ ]:
mf_ndcg_model, mf_ndcg_predictions = train_and_predict_best(
    model_class=MatrixFactorizationSGD,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [75],
        'learning_rate': [0.01],
        'n_epochs': [20],
        'use_bias': [True],
        'eval_every': [30],
        'batch_size': [1024],
        'device': ['auto'],
        'l2_reg': [0.1]
    },
    output_filepath='../outputs/preds/predictions_mf_ndcg.csv',
    metric='ndcg',
    model_name='Matrix Factorization (NDCG-optimized)'
)

print("="*80)

mf_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=mf_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {mf_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {mf_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {mf_ndcg_ranking_metrics['ndcg@10']:.4f}")


## Bayesian Personalized Ranking

BPR trains on user-positive-negative item triples and is designed for personalized ranking. The implementation still exposes rating-style predictions so it can be compared in the shared evaluation tables.


In [ ]:
bpr_model, bpr_predictions = train_and_predict_best(
    model_class=BayesianProbabilisticRanking,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [50, 70],
        'learning_rate': [0.001, 0.02],
        'n_epochs': [30],
        'n_samples': [5],
        'eval_every': [5],
        'batch_size': [512],
        'device': ['auto'],
        'l2_reg': [0.1]
    },
    output_filepath='../outputs/preds/predictions_bpr_best.csv',
    metric='rmse'
)

bpr_model.plot_training_history(figsize=(12, 6), save_path=None)
optimal_epochs_bpr = bpr_model.get_best_epoch()

should_retrain = (
    optimal_epochs_bpr is not None and 
    optimal_epochs_bpr < bpr_model.n_epochs
)

if should_retrain:
    print("\n" + "="*80)
    print("STEP 3: Retraining BPR with optimal number of epochs")
    print("="*80)
    print(f"Optimal epoch found: {optimal_epochs_bpr}")
    print(f"Best validation RMSE: {bpr_model.best_val_rmse:.4f}")
    print(f"\nRetraining model with {optimal_epochs_bpr} epochs...")

    bpr_final = BayesianProbabilisticRanking(
        n_factors=bpr_model.n_factors,
        learning_rate=bpr_model.learning_rate,
        n_epochs=optimal_epochs_bpr,
        n_samples=bpr_model.n_samples,
        eval_every=bpr_model.eval_every,
        batch_size=bpr_model.batch_size,
        l2_reg=bpr_model.l2_reg,
        device='auto'
    ).fit(train_data, val_data)

    print("\n" + "="*80)
    print("BPR Training completed!")
    print("="*80)
    print(f"Final model trained with {optimal_epochs_bpr} epochs")
    print(f"Validation RMSE: {bpr_final.best_val_rmse:.4f}")
    print("="*80)
else:
    print("\n" + "="*80)
    print("No retraining needed - using existing model")
    print("="*80)
    bpr_final = bpr_model


### BPR Ranking Configuration

This sweep directly optimizes ranking metrics and later contributes predictions to the optimized hybrid ranking model.


In [ ]:
bpr_ndcg_model, bpr_ndcg_predictions = train_and_predict_best(
    model_class=BayesianProbabilisticRanking,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    param_grid={
        'n_factors': [32],
        'learning_rate': [0.01],
        'n_epochs': [30],
        'n_samples': [5],
        'eval_every': [40],
        'batch_size': [8192],
        'device': ['auto'],
        'l2_reg': [0.01]
    },
    output_filepath='../outputs/preds/predictions_bpr_ndcg.csv',
    metric='ndcg',
    model_name='BPR (NDCG-optimized)'
)


print("\n" + "="*80)
print("EVALUATING NDCG-OPTIMIZED BPR MODEL")
print("="*80)

bpr_ndcg_ranking_metrics = evaluate_ranking_metrics(
    model=bpr_ndcg_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=1.0
)

print(f"\nRanking Performance (NDCG-optimized):")
print(f"  Precision@10: {bpr_ndcg_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {bpr_ndcg_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {bpr_ndcg_ranking_metrics['ndcg@10']:.4f}")



## Hybrid Rating Model

The rating hybrid learns Ridge-regression weights over validation predictions from content, UserKNN, ItemKNN, and matrix factorization. This lets the final predictor combine complementary error patterns instead of relying on uniform averaging.


In [ ]:
content_val_preds = save_predictions_to_csv(
    model=content_model_best,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_content_val.csv',
    model_name='Content'
)

user_knn_val_preds = save_predictions_to_csv(
    model=user_knn_model,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_userknn_val.csv',
    model_name='UserKNN'
)

item_knn_val_preds = save_predictions_to_csv(
    model=item_knn_model,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_itemknn_val.csv',
    model_name='ItemKNN'
)

mf_val_preds = save_predictions_to_csv(
    model=mf_model,
    test_data=val_data,
    output_filepath='../outputs/preds/predictions_mf_val.csv',
    model_name='MF'
)

hybrid_model = HybridRecommender(
    models={
        'Content': content_model_best,
        'UserKNN': user_knn_model,
        'ItemKNN': item_knn_model,
        'MF': mf_model},
    prediction_files={
        'Content': '../outputs/preds/predictions_content_val.csv',
        'UserKNN': '../outputs/preds/predictions_userknn_val.csv',
        'ItemKNN': '../outputs/preds/predictions_itemknn_val.csv',
        'MF': '../outputs/preds/predictions_mf_val.csv'},
    alpha=1.0
)

hybrid_model.fit(train_data, validation_data=val_data)


print("\n" + "="*80)
print("LEARNED WEIGHTS:")
print("="*80)
weights_df = hybrid_model.get_feature_importance()
display(weights_df)

print("\n" + "="*80)
print("GENERATING PREDICTIONS ON TEST SET")
print("="*80)

hybrid_predictions = save_predictions_to_csv(
    model=hybrid_model,
    test_data=test_data,
    output_filepath='../outputs/preds/predictions_hybrid.csv',
    model_name='Hybrid Recommender'
)

print("\n" + "="*80)
print("EVALUATION")
print("="*80)
hybrid_rmse = evaluate_rmse(hybrid_model, test_data)
print(f"Hybrid Model RMSE: {hybrid_rmse:.4f}")


## Optimized Hybrid Ranking Model

The ranking hybrid merges full candidate-score files, normalizes scores per user, and searches model weights that maximize validation NDCG@10 before precomputing top-k recommendations.


In [ ]:
prediction_files = {
    'Content': '../outputs/preds/all_predictions_content.csv',
    'UserKNN': '../outputs/preds/all_predictions_userknn.csv',
    'ItemKNN': '../outputs/preds/all_predictions_itemknn.csv',
    'MF':      '../outputs/preds/all_predictions_mf.csv',
    'BPR':     '../outputs/preds/all_predictions_bpr.csv',
}

hybrid = OptimizedHybridRanking(
    prediction_files=prediction_files,
    train_data=train_data,
    relevance_threshold=3.0,
    normalize_per_user=True,
    n_dirichlet=300,
    coord_iters=10,
    k_eval=10,
)

best_val_ndcg = hybrid.fit_weights(val_data)

hybrid.precompute_topk()
metrics = evaluate_ranking_metrics(
    model=hybrid,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)
print(metrics)
print(hybrid.get_weights())


## Baseline Models

The baselines define minimum useful comparisons: item-average and mean-hybrid predictors for ratings, plus random and popularity recommenders for ranking.


In [ ]:
print("\n" + "="*80)
print("TRAINING BASELINE MODELS")
print("="*80)
print("\n--- Rating Prediction Baselines ---")

print("\n1. Item Average Baseline")
item_avg_model = ItemAverageBaseline()
item_avg_model.fit(train_data)

item_avg_predictions = save_predictions_to_csv(
    model=item_avg_model,
    test_data=test_data,
    output_filepath='../outputs/preds/predictions_item_average.csv',
    model_name='Item Average Baseline'
)

print("\n2. Mean Hybrid Baseline (Rating)")
mean_hybrid_rating = MeanHybridRating(
    models={
        'UserKNN': user_knn_model,
        'ItemKNN': item_knn_model,
        'MatrixFactorization': mf_model}
)
mean_hybrid_rating.fit(train_data)

mean_hybrid_predictions = save_predictions_to_csv(
    model=mean_hybrid_rating,
    test_data=test_data,
    output_filepath='../outputs/preds/predictions_mean_hybrid.csv',
    model_name='Mean Hybrid Baseline')


In [ ]:
print("\n" + "-"*80)
print("1. RANDOM RECOMMENDER")
print("-"*80)
random_model = RandomRecommender(seed=1)
random_model.fit(train_data)
a = random_model.recommend_topk(user_id=0, n=10)
print(a)
random_ranking_metrics = evaluate_ranking_metrics(
    model=random_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)

print(f"\nRandom Recommender - Ranking Performance:")
print(f"  Precision@10: {random_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {random_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {random_ranking_metrics['ndcg@10']:.4f}")


print("\n" + "-"*80)
print("2. POPULARITY RECOMMENDER")
print("-"*80)
popularity_model = PopularityRecommender()
popularity_model.fit(train_data)
popularity_ranking_metrics = evaluate_ranking_metrics(
    model=popularity_model,
    test_data=test_data,
    train_data=train_data,
    n=10,
    relevance_threshold=3.0
)

print(f"\nPopularity Recommender - Ranking Performance:")
print(f"  Precision@10: {popularity_ranking_metrics['precision@10']:.4f}")
print(f"  Recall@10:    {popularity_ranking_metrics['recall@10']:.4f}")
print(f"  NDCG@10:      {popularity_ranking_metrics['ndcg@10']:.4f}")


## Rating Prediction Results

The learned rating hybrid achieved the best RMSE and MAE, improving over matrix factorization and the mean-hybrid baseline.


| **Model**                               |  **RMSE**  |   **MAE**  |
| :-------------------------------------- | :--------: | :--------: |
| **Content-Based**                       |   1.1762   |   0.9111   |
| **Bayesian Personalized Ranking (BPR)** |   1.1554   |   0.9063   |
| **Item Average Baseline**               |   1.0331   |   0.8277   |
| **Item-Based CF**                       |   1.0064   |   0.7863   |
| **User-Based CF**                       |   0.9675   |   0.7564   |
| **Matrix Factorization (MF)**           |   0.9403   |   0.7427   |
| **Mean Hybrid Baseline**                |   0.9399   |   0.7393   |
| **Hybrid Model (Rating)**               | **0.9250** | **0.7285** |


## Ranking Results

The optimized ranking hybrid produced the strongest top-k performance, substantially improving NDCG@10 over individual recommenders and popularity.


| **Model**                               | **Precision@10** | **Recall@10** | **NDCG@10** |
| :-------------------------------------- | :--------------: | :-----------: | :---------: |
| **Random Recommender**                  |      0.0161      |     0.0067    |    0.0114   |
| **Content-Based**                       |      0.1046      |     0.0386    |    0.0955   |
| **User-Based CF**                       |      0.0107      |     0.0023    |    0.1041   |
| **Matrix Factorization (MF)**           |      0.0791      |     0.0225    |    0.1443   |
| **Item-Based CF**                       |      0.0266      |     0.0030    |    0.1627   |
| **Bayesian Personalized Ranking (BPR)** |      0.2137      |     0.0930    |    0.1778   |
| **Popularity Recommender**              |      0.2231      |     0.1048    |    0.1842   |
| **Hybrid Model (Ranking)**              |    **0.3194**    |   **0.1224**  |  **0.3100** |


## Hybrid Weight Interpretation

The learned weights show which component models contributed most to each objective. Rating prediction leaned most on MF and ItemKNN, while ranking assigned most of its mass to BPR.


| Model   | Weight (Rating) | Weight (Ranking) |
| ------- | --------------: | ---------------: |
| Content |         -0.1498 |           0.0870 |
| UserKNN |          0.1938 |           0.0270 |
| ItemKNN |          0.2853 |           0.1122 |
| MF      |          0.3585 |           0.0887 |
| BPR     |               — |           0.6851 |


| Model   | Rating | Ranking |
| ------- | -----: | ------: |
| Content | 15.17% |   8.70% |
| UserKNN | 19.63% |   2.70% |
| ItemKNN | 28.90% |  11.22% |
| MF      | 36.30% |   8.87% |
| BPR     |      — |  68.51% |


## Persisting Trained Models

Persistence utilities are imported from the package. The notebook only assembles the trained model dictionary and writes ignored artifacts to `outputs/models`.


In [ ]:
trained_models = {
    'content_rmse': content_model_best,
    'content_ndcg': content_ndcg_model,
    'userknn_rmse': user_knn_model,
    'userknn_ndcg': user_knn_ndcg_model,
    'itemknn_rmse': item_knn_model,
    'itemknn_ndcg': item_knn_ndcg_model,
    'mf_rmse': mf_model,
    'mf_ndcg': mf_ndcg_model,
    'bpr_rmse': bpr_model,
    'bpr_ndcg': bpr_ndcg_model,
    'hybrid_rating': hybrid_model,
    'hybrid_ranking': hybrid,
}

save_all_models(trained_models, models_dir='../outputs/models')
loaded_models = load_all_models(models_dir='../outputs/models')
print(f"Loaded {len(loaded_models)} persisted models.")
